# 3 · Class nodes and what a run supplies

Helpers live on the class, and every call gets a fresh instance. What only the caller has (who runs the graph, a clock, a client) is `Annotated[T, FromRun()]`, handed in as `compiled.run(from_run={T: value})`.

In [ ]:
from typing import Annotated

from conductor import CompiledGraph, From, FromRun, Graph, GraphNode, model_of, NodeRegistry, Param, Result, Static, node
from conductor.widgets import Textarea
from conductor_nodes.types import Number, Text

registry = NodeRegistry()

## A class node

In [ ]:
from conductor import node
def _words(text: str) -> list[str]:
    return text.split()

@node("word-count", title="Word count", description="Counts the words in a text", category="text")
def word_count(text: Annotated[Text, Param(title="Text", widget=Textarea())]) -> Annotated[Number, Result(title="Words")]:
    return Number(len(_words(text)))


registry = registry.extended_with([word_count])
word_count(text=Text("the quick brown fox"))

## A value the run supplies

`Caller` is the host's type. The parameter has no widget, no handle and no binding.

In [ ]:
from conductor import node
class Caller:
    """Whoever is running the graph — a type the host declares."""

    def __init__(self, name: str) -> None:
        self.name = name


@node("signed", title="Signed", description="Signs a text with the name of whoever ran the graph", category="text")
def signed(
    text: Annotated[Text, Param(title="Text", widget=Textarea())],
    who: FromRun[Caller],
) -> Annotated[Text, Result(title="Signed")]:
    return Text(f"{text} — {who.name}")


registry = registry.extended_with([signed])

## A need, not an input

`Interface.needs` lists it; the inputs and the check a call goes through do not.

In [ ]:
interface = signed.versions[1].interface

print("inputs:", [inp.name for inp in interface.inputs])
print("needs: ", sorted(needed.__name__ for needed in interface.needs))
assert "who" not in model_of(interface.inputs)({"text": "hej", "who": "Ida"})

## Run it

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="note", type="signed", version=1, bindings={"text": Static("The quick brown fox jumps over the lazy dog")}),
        GraphNode(id="count", type="word-count", version=1, bindings={"text": From("note.result")}),
    ]),
    registry,
)

results = compiled.results(await compiled.run(from_run={Caller: Caller("Ida")}))
print(results["note"]["result"])
print(results["count"]["result"])

A leg missing a need refuses to start:

In [ ]:
try:
    await compiled.run()
except TypeError as refused:
    print(refused)